# Evaluación Parcial 1 - Machine Learning
**Asignatura:** MLY0100 Machine Learning  
**Integrantes:** Christian Sandoval, Nicolás Vega  
**Fecha:** 25 de septiembre de 2026  
**Dataset:** `DS1-18-Datos-Properati.csv`

# Fase 1 - Comprensión del Negocio

En este trabajo usamos un dataset de propiedades en venta de Argentina. La idea es revisar los datos, entenderlos y dejarlos preparados para poder usarlos más adelante en Machine Learning.

**Objetivo:** identificar patrones en los precios de las propiedades y preparar un dataset limpio para una etapa posterior de modelado.

**Pregunta de negocio:** ¿Qué características de las propiedades, como la ubicación, la superficie y el tipo de propiedad, se relacionan con las diferencias de precio?

**Supuestos:**
- Los valores nulos no significan cero.
- Los valores muy altos se revisan antes de modificarlos.
- Las conclusiones corresponden solamente a este dataset.

**Target para regresión:** `price`, porque es una variable numérica continua y representa una cantidad que podría predecirse.

**Target para clasificación:** `property_type`, porque contiene clases discretas como Departamento, Casa y PH.

En esta entrega no se entrenan modelos. Solo se definen los targets y se trabajan las tres primeras fases de CRISP-DM.


# Fase 2 - Comprensión de los Datos

## Carga e inspección

Primero importamos las librerías que vamos a utilizar. Luego cargamos el CSV que viene dentro del archivo ZIP. Revisamos `shape`, `head()`, tipos de datos, `info()` y `describe()` para conocer la estructura general del dataset antes de tomar decisiones de limpieza.


In [ ]:
%matplotlib inline

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Nombre del archivo entregado
nombre_zip = 'Evaluación Parcial 1 - Dataset.zip'

# Buscamos el ZIP en la carpeta actual o en la carpeta anterior
if os.path.exists(nombre_zip):
    ruta_zip = nombre_zip
else:
    ruta_zip = '../' + nombre_zip

# Si estamos en Colab y no está el ZIP, lo subimos
if not os.path.exists(ruta_zip):
    from google.colab import files
    files.upload()
    ruta_zip = nombre_zip

# Abrimos el ZIP y cargamos el CSV
with zipfile.ZipFile(ruta_zip) as archivo_zip:
    nombre_csv = [
        nombre for nombre in archivo_zip.namelist()
        if nombre.endswith('DS1-18-Datos-Properati.csv')
        and not nombre.startswith('__MACOSX')
    ][0]

    with archivo_zip.open(nombre_csv) as archivo_csv:
        df = pd.read_csv(archivo_csv)

print('Dimensiones:', df.shape)
print(df.dtypes)
df.info()

display(df.head())
display(df.describe(include='all').T)


El dataset tiene **146.660 filas y 19 columnas**. Las fechas aparecen inicialmente como texto. También hay valores faltantes en `lat`, `lon`, `bathrooms`, `surface_total` y `surface_covered`, por lo que esas variables deberán tratarse en la preparación.

## Estadísticos descriptivos

Para las variables numéricas relevantes calculamos tendencia central y dispersión. Partimos desde `describe()` para reunir los estadísticos principales y agregamos moda, varianza e IQR. Esto permite comparar el valor típico de las propiedades con la dispersión y detectar distribuciones muy asimétricas.


In [ ]:
variables = [
    'rooms',
    'bedrooms',
    'bathrooms',
    'surface_total',
    'surface_covered',
    'price'
]

# La tabla parte desde describe(), como resumen estadístico principal
estadisticos = df[variables].describe().T
estadisticos['moda'] = df[variables].mode().iloc[0]
estadisticos['varianza'] = df[variables].var()
estadisticos['IQR'] = estadisticos['75%'] - estadisticos['25%']

estadisticos = estadisticos.rename(columns={
    'mean': 'media',
    '50%': 'mediana',
    'std': 'desviacion_estandar'
})

columnas_tabla = [
    'media',
    'mediana',
    'moda',
    'desviacion_estandar',
    'varianza',
    '25%',
    '75%',
    'IQR'
]

display(estadisticos[columnas_tabla].round(2))


En `price`, `surface_total` y `surface_covered` la media queda bastante por encima de la mediana. Esto muestra asimetría hacia valores altos y presencia de valores extremos. En `price`, por ejemplo, la mediana representa mejor el precio típico que la media porque la media recibe más influencia de las propiedades muy costosas.

## Distribuciones y relación con el negocio

Ahora usamos histogramas, boxplots, barras, scatter y heatmap. Para algunas visualizaciones usamos el percentil 99 solamente para que unos pocos valores muy altos no deformen los gráficos. Además de revisar frecuencias, comparamos el **precio mediano por tipo de propiedad y por zona**, porque ambas variables forman parte de la pregunta de negocio.


In [ ]:
p99_price = df['price'].quantile(0.99)
p99_superficie = df['surface_total'].quantile(0.99)

precios = df.loc[df['price'] <= p99_price, 'price'].dropna()

# Histograma de precio
plt.figure(figsize=(8, 5))
plt.hist(precios, bins=40, edgecolor='black')
plt.axvline(precios.mean(), linestyle='--', label='Media')
plt.axvline(precios.median(), linestyle='-', label='Mediana')
plt.title('Distribución del precio')
plt.xlabel('Precio (USD)')
plt.ylabel('Cantidad')
plt.legend()
plt.show()

# Boxplots de precio y superficie
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].boxplot(precios, vert=False)
ax[0].set_title('Boxplot de price')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Distribución')

superficies = df.loc[
    df['surface_total'] <= p99_superficie,
    'surface_total'
].dropna()

ax[1].boxplot(superficies, vert=False)
ax[1].set_title('Boxplot de surface_total')
ax[1].set_xlabel('Superficie total (m²)')
ax[1].set_ylabel('Distribución')
plt.tight_layout()
plt.show()

# Tipos de propiedad con más publicaciones
tipos = df['property_type'].value_counts().head(10)
plt.figure(figsize=(9, 5))
plt.barh(tipos.index, tipos.values)
plt.title('10 tipos de propiedad con más publicaciones')
plt.xlabel('Cantidad')
plt.ylabel('Tipo de propiedad')
plt.gca().invert_yaxis()
plt.show()

# Precio mediano de los tipos de propiedad con más registros
tipos_principales = df['property_type'].value_counts().head(6).index
mediana_precio_tipo = (
    df[df['property_type'].isin(tipos_principales)]
    .groupby('property_type', observed=False)['price']
    .median()
    .sort_values()
)

plt.figure(figsize=(9, 5))
plt.barh(mediana_precio_tipo.index, mediana_precio_tipo.values)
plt.title('Precio mediano por tipo de propiedad')
plt.xlabel('Precio mediano (USD)')
plt.ylabel('Tipo de propiedad')
plt.show()

# Precio mediano por zona
mediana_precio_zona = (
    df.groupby('l2', observed=False)['price']
    .median()
    .sort_values()
)

plt.figure(figsize=(9, 5))
plt.barh(mediana_precio_zona.index, mediana_precio_zona.values)
plt.title('Precio mediano por zona')
plt.xlabel('Precio mediano (USD)')
plt.ylabel('Zona')
plt.show()

# Relación entre superficie y precio
muestra = df[
    (df['surface_total'] <= p99_superficie) &
    (df['price'] <= p99_price)
][['surface_total', 'price']].dropna()

muestra = muestra.sample(5000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(muestra['surface_total'], muestra['price'], alpha=0.35)
plt.title('Superficie total y precio')
plt.xlabel('Superficie total (m²)')
plt.ylabel('Precio (USD)')
plt.show()

# Matriz de correlación
plt.figure(figsize=(9, 6))
sns.heatmap(df[variables].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.show()

print('Precio mediano por tipo de propiedad:')
display(mediana_precio_tipo.sort_values(ascending=False).to_frame('precio_mediano'))

print('Precio mediano por zona:')
display(mediana_precio_zona.sort_values(ascending=False).to_frame('precio_mediano'))


Los gráficos muestran que el precio es asimétrico y que la superficie total tiene una relación positiva con el precio, aunque existe bastante dispersión. En los tipos más frecuentes, por ejemplo, la mediana de `Casa` es aproximadamente **USD 210.000**, mientras `Departamento` y `PH` quedan cerca de **USD 160.000**. Por zona, **Bs.As. G.B.A. Zona Norte** tiene una mediana cercana a **USD 185.000**, mientras **Zona Oeste** queda cerca de **USD 113.000**. Por eso la superficie ayuda a explicar el precio, pero la ubicación y el tipo de propiedad también se relacionan con sus diferencias.

# Fase 3 - Preparación de los Datos

## Missing values

Primero contamos los nulos y revisamos su porcentaje. Para analizar el mecanismo de ausencia comprobamos si el porcentaje de `surface_total` cambia según `property_type` y si `lat` y `lon` faltan juntos. Si la ausencia cambia según variables observadas, el patrón es **compatible con MAR**. Lo tomamos como una hipótesis de trabajo, no como una demostración definitiva.

También probamos `KNNImputer` sobre una muestra de 1.500 registros. Usamos solo variables numéricas porque KNN calcula distancias entre observaciones. `n_neighbors=2` indica que utiliza dos vecinos y `weights='uniform'` significa que ambos vecinos tienen el mismo peso.


In [ ]:
# Cantidad y porcentaje de nulos
resumen_nulos = pd.DataFrame({
    'cantidad_nulos': df.isna().sum(),
    'porcentaje_nulos': (df.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos[
    resumen_nulos['cantidad_nulos'] > 0
].sort_values('porcentaje_nulos', ascending=False)

display(resumen_nulos)

plt.figure(figsize=(8, 5))
plt.barh(resumen_nulos.index, resumen_nulos['porcentaje_nulos'])
plt.title('Porcentaje de valores faltantes')
plt.xlabel('Porcentaje (%)')
plt.ylabel('Variable')
plt.gca().invert_yaxis()
plt.show()

# Evidencia para la hipótesis MAR: nulos de superficie según tipo de propiedad
nulos_por_tipo = (
    df.groupby('property_type', observed=False)['surface_total']
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

print('Porcentaje de surface_total faltante según property_type:')
display(nulos_por_tipo.to_frame('porcentaje_nulos'))

# Revisamos si lat y lon faltan juntos
faltan_lat_lon = (df['lat'].isna() & df['lon'].isna()).sum()
print('Nulos en lat:', int(df['lat'].isna().sum()))
print('Nulos en lon:', int(df['lon'].isna().sum()))
print('Filas donde faltan lat y lon:', int(faltan_lat_lon))

# Probamos KNNImputer con una muestra para que no demore tanto
muestra_knn = df[
    ['bathrooms', 'surface_total', 'surface_covered']
].sample(1500, random_state=42)

datos_knn = muestra_knn.copy()

print('Nulos antes de KNN:', int(muestra_knn.isna().sum().sum()))

knn_imputer = KNNImputer(
    n_neighbors=2,
    weights='uniform'
)

datos_knn[
    ['bathrooms', 'surface_total', 'surface_covered']
] = knn_imputer.fit_transform(
    datos_knn[['bathrooms', 'surface_total', 'surface_covered']]
)

print('Nulos después de KNN:', int(datos_knn.isna().sum().sum()))

comparacion_knn = pd.DataFrame({
    'media_original': muestra_knn.mean(),
    'media_knn': datos_knn.mean(),
    'std_original': muestra_knn.std(),
    'std_knn': datos_knn.std()
})

display(comparacion_knn.round(2))

# Para la limpieza final usamos medianas por grupo
df_limpio = df.copy()

for columna in ['surface_total', 'surface_covered', 'bathrooms']:
    mediana = df_limpio.groupby(
        'property_type',
        observed=False
    )[columna].transform('median')
    df_limpio[columna] = df_limpio[columna].fillna(mediana)

for columna in ['lat', 'lon']:
    mediana = df_limpio.groupby(
        'l3',
        observed=False
    )[columna].transform('median')
    df_limpio[columna] = df_limpio[columna].fillna(mediana)

# Comparamos el efecto de la imputación final
columnas_imputadas = ['bathrooms', 'surface_total', 'surface_covered']
comparacion_imputacion = pd.DataFrame({
    'media_antes': df[columnas_imputadas].mean(),
    'media_despues': df_limpio[columnas_imputadas].mean(),
    'mediana_antes': df[columnas_imputadas].median(),
    'mediana_despues': df_limpio[columnas_imputadas].median(),
    'std_antes': df[columnas_imputadas].std(),
    'std_despues': df_limpio[columnas_imputadas].std()
})

display(comparacion_imputacion.round(2))
print('Nulos restantes:', int(df_limpio.isna().sum().sum()))
print('Filas después de imputar:', len(df_limpio))


La cantidad de faltantes cambia según variables observadas y `lat` y `lon` presentan un patrón conjunto: hay **9.925 filas** donde ambas coordenadas faltan a la vez. Por eso tratamos la ausencia como **principalmente compatible con MAR**. En la muestra de 1.500 registros, KNN pasa de **449 nulos a 0**. Para el dataset completo elegimos **mediana agrupada** porque es simple de interpretar y se ve menos afectada por valores extremos. La media de `bathrooms` pasa aproximadamente de 1,60 a 1,59; la de `surface_total` de 216,87 a 208,04; y la de `surface_covered` de 112,82 a 110,10, por lo que los valores centrales no cambian de forma brusca.

No usamos `dropna()` como tratamiento general porque superficies y coordenadas tienen porcentajes de ausencia suficientemente altos como para perder información útil si elimináramos todas esas filas.

## Outliers

Detectamos valores atípicos con **IQR y Z-score** y los visualizamos con boxplots. Como IQR marca muchos valores altos que podrían corresponder a propiedades reales, no eliminamos todos esos casos. Para el tratamiento consideramos como extremos los valores ubicados sobre el percentil 99 de las variables numéricas relevantes y eliminamos esas filas. Luego informamos exactamente cuántas observaciones se pierden.


In [ ]:
# Detección de outliers con IQR y Z-score
lista_outliers = []

for columna in variables:
    q1 = df_limpio[columna].quantile(0.25)
    q3 = df_limpio[columna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mascara_iqr = (
        (df_limpio[columna] < limite_inferior) |
        (df_limpio[columna] > limite_superior)
    )

    z_score = np.abs(stats.zscore(df_limpio[columna]))

    lista_outliers.append([
        columna,
        int(mascara_iqr.sum()),
        round(mascara_iqr.mean() * 100, 2),
        int((z_score > 3).sum()),
        round((z_score > 3).mean() * 100, 2)
    ])

resumen_outliers = pd.DataFrame(
    lista_outliers,
    columns=[
        'variable',
        'outliers_iqr',
        'porcentaje_iqr',
        'outliers_zscore',
        'porcentaje_zscore'
    ]
)

display(resumen_outliers)

# Boxplots antes del tratamiento
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].boxplot(df_limpio['price'], vert=False)
ax[0].set_title('Price antes del tratamiento')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Distribución')

ax[1].boxplot(df_limpio['surface_total'], vert=False)
ax[1].set_title('Surface_total antes del tratamiento')
ax[1].set_xlabel('Superficie total (m²)')
ax[1].set_ylabel('Distribución')
plt.tight_layout()
plt.show()

# Eliminamos solamente filas con valores superiores al percentil 99
mascara_extremos = np.zeros(len(df_limpio), dtype=bool)
limites_p99 = []

for columna in variables:
    limite_p99 = df_limpio[columna].quantile(0.99)
    mascara_columna = df_limpio[columna] > limite_p99
    mascara_extremos = mascara_extremos | mascara_columna.to_numpy()
    limites_p99.append([
        columna,
        limite_p99,
        int(mascara_columna.sum())
    ])

resumen_limites = pd.DataFrame(
    limites_p99,
    columns=['variable', 'limite_p99', 'casos_sobre_p99']
)

display(resumen_limites.round(2))

filas_antes_outliers = len(df_limpio)
df_limpio = df_limpio.loc[~mascara_extremos].copy()
filas_eliminadas_outliers = filas_antes_outliers - len(df_limpio)
porcentaje_eliminado = filas_eliminadas_outliers / filas_antes_outliers * 100

print('Filas antes:', filas_antes_outliers)
print('Filas eliminadas por extremos:', filas_eliminadas_outliers)
print('Porcentaje eliminado:', round(porcentaje_eliminado, 2), '%')
print('Filas después:', len(df_limpio))

# Boxplots después del tratamiento
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].boxplot(df_limpio['price'], vert=False)
ax[0].set_title('Price después del tratamiento')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Distribución')

ax[1].boxplot(df_limpio['surface_total'], vert=False)
ax[1].set_title('Surface_total después del tratamiento')
ax[1].set_xlabel('Superficie total (m²)')
ax[1].set_ylabel('Distribución')
plt.tight_layout()
plt.show()


IQR detecta muchos más casos en las variables muy asimétricas, mientras que Z-score identifica un conjunto menor de observaciones extremas. Para no eliminar todos los valores altos del mercado, el tratamiento se limita a filas que superan el **percentil 99** en alguna variable numérica relevante. Con este criterio se eliminan **5.200 filas**, equivalentes al **3,55 %** del dataset, y quedan **141.460 registros** para continuar la preparación.

## Normalización y estandarización

Después del tratamiento de outliers revisamos la asimetría de cada variable numérica relevante. `rooms` y `bedrooms` son las menos asimétricas, por eso usamos **StandardScaler**. `bathrooms`, `surface_total`, `surface_covered` y `price` siguen siendo más asimétricas, por lo que usamos **MinMaxScaler** para llevarlas al rango 0 a 1. También revisamos `lat` y `lon`, pero **no se escalan** porque representan coordenadas geográficas y en esta etapa se conservan en su unidad original para mantener una interpretación directa de la ubicación.

| Decisión | Técnica | Motivo |
| --- | --- | --- |
| Valores faltantes | Mediana agrupada | Conserva registros y reduce el efecto de valores extremos |
| Outliers extremos | Eliminación sobre p99 | Evita borrar todos los valores altos detectados por IQR |
| `rooms` y `bedrooms` | StandardScaler | Son las variables menos asimétricas |
| Baños, superficies y precio | MinMaxScaler | Mantienen mayor asimetría y se llevan a 0-1 |
| `lat` y `lon` | No se escalan | Son coordenadas geográficas y se mantienen interpretables |

> **Decisión:** aplicamos cada tratamiento según la naturaleza observada de la variable y comprobamos sus resultados antes de continuar.


In [ ]:
# Revisamos la asimetría después de tratar los outliers
variables_escalamiento = variables + ['lat', 'lon']
asimetria = df_limpio[variables_escalamiento].skew().round(3)

decision_escalamiento = pd.DataFrame({
    'asimetria': asimetria,
    'tecnica': 'MinMaxScaler'
})

decision_escalamiento.loc[
    ['rooms', 'bedrooms'],
    'tecnica'
] = 'StandardScaler'

decision_escalamiento.loc[
    ['lat', 'lon'],
    'tecnica'
] = 'No se escala'

display(decision_escalamiento)

columnas_standard = ['rooms', 'bedrooms']
columnas_minmax = [
    'bathrooms',
    'surface_total',
    'surface_covered',
    'price'
]

df_escalado = df_limpio.copy()

# Estandarización
scaler_standard = StandardScaler()
datos_standard = scaler_standard.fit_transform(
    df_limpio[columnas_standard]
)

df_escalado[
    [columna + '_std' for columna in columnas_standard]
] = datos_standard

# Normalización
scaler_minmax = MinMaxScaler()
datos_minmax = scaler_minmax.fit_transform(
    df_limpio[columnas_minmax]
)

df_escalado[
    [columna + '_minmax' for columna in columnas_minmax]
] = datos_minmax

print('Antes:')
display(
    df_limpio[variables]
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(3)
)

print('Después de StandardScaler:')
display(
    df_escalado[
        [columna + '_std' for columna in columnas_standard]
    ]
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(3)
)

print('Después de MinMaxScaler:')
display(
    df_escalado[
        [columna + '_minmax' for columna in columnas_minmax]
    ]
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(3)
)


Después de `StandardScaler`, las variables estandarizadas quedan con media cercana a 0 y desviación estándar cercana a 1. Después de `MinMaxScaler`, las variables normalizadas quedan entre 0 y 1. La comparación antes y después confirma que las transformaciones se aplicaron como se esperaba.

## Dataset final y exportación

Para terminar corregimos los tipos de datos. Las fechas se convierten a `datetime` y las variables categóricas a `category`. Finalmente comprobamos dimensiones, nulos y tipos, y exportamos el dataset preparado con `to_csv()`.


In [ ]:
df_final = df_escalado.copy()

# Fechas
for columna in ['start_date', 'end_date', 'created_on']:
    df_final[columna] = df_final[columna].astype(
        'datetime64[s]'
    )

# Variables categóricas
for columna in [
    'l1',
    'l2',
    'l3',
    'currency',
    'property_type',
    'operation_type'
]:
    df_final[columna] = df_final[columna].astype('category')

print('Dimensiones finales:', df_final.shape)
print('Valores nulos:', int(df_final.isna().sum().sum()))
print(df_final.dtypes)

df_final.to_csv(
    'DS1-18-Datos-Properati-preparado.csv',
    index=False
)

print('Dataset exportado correctamente.')


# Conclusiones

En este trabajo completamos las tres primeras fases de CRISP-DM: comprensión del negocio, comprensión de los datos y preparación de los datos.

La pregunta de negocio buscaba conocer qué características se relacionan con las diferencias de precio. El análisis muestra que **la superficie total se relaciona positivamente con el precio**, aunque existe dispersión. También observamos diferencias en el **precio mediano según el tipo de propiedad** y según la **zona (`l2`)**, por lo que el precio no depende de una sola característica.

En los datos faltantes observamos patrones compatibles principalmente con **MAR**. Probamos `KNNImputer` en una muestra de 1.500 registros y comprobamos que completa sus nulos. Para la limpieza final usamos medianas agrupadas, conservando los registros antes del tratamiento de outliers y terminando esa etapa con 0 nulos.

Los outliers se detectaron con IQR y Z-score. Como muchas propiedades caras pueden ser casos reales, no eliminamos todo lo marcado por IQR. Se eliminaron solamente las filas que superaban el percentil 99 en alguna variable numérica relevante y se informó el porcentaje de registros eliminado.

Después analizamos la asimetría para elegir el escalamiento. Usamos `StandardScaler` en `rooms` y `bedrooms`, y `MinMaxScaler` en `bathrooms`, superficies y `price`. `lat` y `lon` se revisaron, pero no se escalan porque se conservan como coordenadas geográficas interpretables. Finalmente corregimos los tipos de datos y exportamos el CSV preparado.

El dataset final queda con **141.460 filas, 25 columnas y 0 valores nulos**, con los tipos corregidos y los extremos superiores tratados. Como próximo paso, este dataset puede utilizarse para modelado, usando `price` como target de regresión o `property_type` como target de clasificación.
